# Pokemon Deck Ranking - Run All

Notebook operativo per eseguire la pipeline unica su Pocket o Pokemon TCG. La logica vive in `pipelines.deck_ranking.run_deck_ranking`; qui restano solo selezione profilo/config, run e preview dei risultati.


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
if not (BASE_DIR / "config" / "config.yaml").exists():
    BASE_DIR = BASE_DIR.parent

import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from IPython.display import Image, display

from pipelines.deck_ranking import run_deck_ranking
from reporting.tables import (
    analysis_scope_summary_frame,
    candidate_vs_full_summary_frame,
    coverage_volume_summary_frame,
    diagnostics_preview_frame,
    evidence_core_comparison_frame,
    evidence_core_eligibility_frame,
    evidence_core_iterative_frame,
    evidence_core_iterative_summary_frame,
    frame_inventory_frame,
    meta_diagnostics_summary_frame,
    nan_diagnostics_critical_frame,
    output_paths_frame,
    share_distribution_frame,
    show_ranking,
    wildcard_review_frame,
)

# Scegli il profilo da eseguire: "pocket", "tcg" oppure "tcg_wildcard".
GAME_PROFILE = "tcg_wildcard"
CONFIG_BY_PROFILE = {
    "pocket": BASE_DIR / "config" / "config.yaml",
    "tcg": BASE_DIR / "config" / "config_tcg.yaml",
    "tcg_wildcard": BASE_DIR / "config" / "config_tcg_wildcard.yaml",
}

RUN_SCRAPE = True
RUN_MARS = True
RUN_HEATMAP = True
RUN_REPORT = True
HEATMAP_TOP_N = 10
SHOW_PROGRESS = True

print(f"Profilo selezionato: {GAME_PROFILE}")


In [ ]:
selected_profile = GAME_PROFILE.strip().lower()
if selected_profile not in CONFIG_BY_PROFILE:
    valid_profiles = ", ".join(sorted(CONFIG_BY_PROFILE))
    raise ValueError(f"GAME_PROFILE non valido: {GAME_PROFILE!r}. Usa uno tra: {valid_profiles}.")

CONFIG_PATH = CONFIG_BY_PROFILE[selected_profile]
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config non trovato per profilo {selected_profile!r}: {CONFIG_PATH}")

print(f"Eseguo profilo: {selected_profile} | config: {CONFIG_PATH.relative_to(BASE_DIR)}")

result = run_deck_ranking(
    base_dir=BASE_DIR,
    config_path=CONFIG_PATH,
    run_scrape=RUN_SCRAPE,
    run_mars=RUN_MARS,
    run_heatmap=RUN_HEATMAP,
    run_report=RUN_REPORT,
    heatmap_top_n=HEATMAP_TOP_N,
    show_progress=SHOW_PROGRESS,
)

actual_game = str((result.cfg.get("source") or {}).get("game") or "").upper()
expected_game = {"pocket": "POCKET", "tcg": "PTCG", "tcg_wildcard": "PTCG"}[selected_profile]
if actual_game != expected_game:
    raise RuntimeError(
        f"Profilo/config non coerenti: GAME_PROFILE={selected_profile!r}, "
        f"ma source.game={actual_game!r} in {CONFIG_PATH}."
    )


In [ ]:
exp_code = getattr(result.expansion, "code", None)
exp_name = getattr(result.expansion, "name", None)
exp_label = f"{exp_code} - {exp_name}" if exp_code and exp_name else (exp_code or "<AUTO>")
profile = selected_profile if "selected_profile" in globals() else GAME_PROFILE

print("Profilo:", profile)
print("Config:", CONFIG_PATH.relative_to(BASE_DIR))
print("Set:", exp_label)
print("Decks URL:", result.decks_url)
print("Scope output:", " / ".join(result.diagnostics.get("source_scope", [])))

report_path = result.outputs.get("report_latest") or result.outputs.get("report")
if report_path is not None:
    print("Report MARS:", report_path.relative_to(BASE_DIR) if report_path.is_relative_to(BASE_DIR) else report_path)

print("\nOutput scritti:")
display(output_paths_frame(result.outputs, base_dir=BASE_DIR))

print("\nDiagnostiche principali:")
display(diagnostics_preview_frame(result.diagnostics))

decklist_raw = result.frames.get("decklist_raw")
top_meta = result.frames.get("top_meta_decklist")
mars_ranking = result.frames.get("mars_ranking")
score_flat = result.frames.get("score_flat")
nan_diag = result.frames.get("nan_diagnostics_pre_filter")
wildcard_candidates = result.frames.get("wildcard_candidates")
n_dir_matrix = result.frames.get("n_dir_matrix")
matchup_raw = result.frames.get("matchup_raw")

summary = meta_diagnostics_summary_frame(
    top_meta=top_meta,
    nan_diag=nan_diag,
    wildcard_candidates=wildcard_candidates,
)
if not summary.empty:
    print("\nSummary meta/coverage:")
    display(summary)

candidate_share_pct = ((result.cfg.get("analysis") or {}).get("candidate_pool") or {}).get("share_pct")
print("\nScope run: fetch vs core MARS:")
display(
    analysis_scope_summary_frame(
        decklist_raw=decklist_raw,
        top_meta=top_meta,
        mars_ranking=mars_ranking,
        score_flat=score_flat,
        matchup_raw=matchup_raw,
        wildcard_candidates=wildcard_candidates,
        candidate_share_pct=candidate_share_pct,
    )
)

if decklist_raw is not None and top_meta is not None:
    request_delay_sec = float((result.cfg.get("scraping") or {}).get("request_delay_sec", 0.0) or 0.0)
    print("\nCandidate pool vs full decklist:")
    display(
        candidate_vs_full_summary_frame(
            decklist_raw=decklist_raw,
            top_meta=top_meta,
            candidate_share_pct=candidate_share_pct,
            request_delay_sec=request_delay_sec,
        )
    )


if decklist_raw is not None and nan_diag is not None and not nan_diag.empty:
    print("\nEvidence-based core simulation:")
    display(
        evidence_core_comparison_frame(
            decklist_raw=decklist_raw,
            nan_diag=nan_diag,
            share_core_pct=80.0,
            min_coverage_pct=60.0,
            min_total_matches=50.0,
        )
    )
    evidence_preview = evidence_core_eligibility_frame(
        decklist_raw=decklist_raw,
        nan_diag=nan_diag,
        min_coverage_pct=60.0,
        min_total_matches=50.0,
        top_n=30,
    )
    display(evidence_preview)

if decklist_raw is not None and ((matchup_raw is not None and not matchup_raw.empty) or (n_dir_matrix is not None and not n_dir_matrix.empty)):
    print("\nEvidence-based core iterativo:")
    iterative_core = evidence_core_iterative_frame(
        decklist_raw=decklist_raw,
        n_dir_matrix=n_dir_matrix,
        matchup_raw=matchup_raw,
        share_core_pct=80.0,
        min_coverage_vs_axis_pct=60.0,
        min_n_vs_axis=50.0,
    )
    display(evidence_core_iterative_summary_frame(iterative_core))
    display(iterative_core[iterative_core["selected"]].head(40))

if top_meta is not None and not top_meta.empty:
    print("\nDistribuzione share - top candidate:")
    display(share_distribution_frame(top_meta, top_n=15))

if nan_diag is not None and not nan_diag.empty:
    print("\nDistribuzione coverage/NaN/volume:")
    display(coverage_volume_summary_frame(nan_diag))

    critical_nan = nan_diagnostics_critical_frame(nan_diag, top_n=15)
    if critical_nan.empty:
        print("\nDiagnostica NaN pre-filtro: nessun deck critico, coverage completa nella candidate pool.")
    else:
        print("\nDiagnostica NaN pre-filtro - deck con piu' buchi di coverage:")
        display(critical_nan)


In [ ]:
ranking = result.frames.get("mars_ranking")
if ranking is not None and not ranking.empty:
    exp_code = getattr(result.expansion, "code", None) or "<AUTO>"
    game_scope = " / ".join(result.diagnostics.get("source_scope", []))
    show_ranking(
        ranking,
        top_n=min(30, len(ranking)),
        title=f"MARS - Top {min(30, len(ranking))} ({game_scope}, set {exp_code})",
    )
else:
    print("Ranking non disponibile: controlla RUN_MARS o gli output precedenti.")


In [ ]:
heatmap_path = (
    result.outputs.get("heatmap_topN_latest")
    or result.outputs.get("heatmap_topN")
)
if heatmap_path is not None and heatmap_path.exists():
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap non disponibile: controlla RUN_HEATMAP o gli output precedenti.")


## Appendice wildcard

Sezione diagnostica separata dal ranking MARS: mostra i deck esclusi dal core che hanno coverage e volume sufficienti contro il core finale. La lettura separa `evidence_tier` da `performance_tier`: un deck puo' avere dati solidi ma performance mediocre, oppure performance interessante ma evidenza ancora sottile.


In [ ]:
wildcard_candidates = result.frames.get("wildcard_candidates")
score_flat = result.frames.get("score_flat")

if wildcard_candidates is None or wildcard_candidates.empty:
    print("Nessuna wildcard candidata con le soglie attuali.")
else:
    core_wr_median = None
    if score_flat is not None and not score_flat.empty:
        sf = score_flat.copy()
        sf["W"] = sf["W"].astype(float)
        sf["L"] = sf["L"].astype(float)
        core_perf = sf.groupby("Deck A", sort=False)[["W", "L"]].sum()
        denom = core_perf["W"] + core_perf["L"]
        core_perf = core_perf[denom > 0].copy()
        core_perf["WR_vs_core_weighted_%"] = 100.0 * core_perf["W"] / (core_perf["W"] + core_perf["L"])
        core_wr_median = float(core_perf["WR_vs_core_weighted_%"].median()) if not core_perf.empty else None

    wc_review = wildcard_review_frame(
        wildcard_candidates,
        core_wr_baseline=core_wr_median,
        min_coverage_high_confidence=100.0,
        min_n_high_confidence=300.0,
        top_n=None,
    )

    print("Wildcard candidates:", len(wildcard_candidates))
    if core_wr_median is not None:
        print(f"Mediana WR pesata del core: {core_wr_median:.2f}%")

    display(wc_review["promotion_tier"].value_counts().rename_axis("promotion_tier").reset_index(name="count"))
    display(wc_review["evidence_tier"].value_counts().rename_axis("evidence_tier").reset_index(name="count"))
    display(wc_review["performance_tier"].value_counts().rename_axis("performance_tier").reset_index(name="count"))
    display(wc_review.head(40))


In [ ]:
# Accesso rapido ai DataFrame principali
decklist_raw = result.frames.get("decklist_raw")
top_meta_decklist = result.frames.get("top_meta_decklist")
matchup_raw = result.frames.get("matchup_raw")
score_flat = result.frames.get("score_flat")
wr_matrix = result.frames.get("wr_matrix")
n_dir_matrix = result.frames.get("n_dir_matrix")
nan_diagnostics_pre_filter = result.frames.get("nan_diagnostics_pre_filter")
mars_ranking = result.frames.get("mars_ranking")

display(frame_inventory_frame(result.frames))
